# Exploración Bronze - Generación EERSA 2021
Carga los Parquet de Bronze con DuckDB y valida calidad básica.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Cargar todos los Parquet de Bronze
df = con.execute("""
    SELECT * FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
""").df()

print(f"Filas totales: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print(f"Rango fechas: {df['fecha'].min()} → {df['fecha'].max()}")
df.head()

## Conteo por planta

In [ ]:
con.execute("""
    SELECT planta, metrica, COUNT(*) as n, 
           COUNT(valor) as no_nulos,
           ROUND(AVG(valor), 2) as promedio,
           ROUND(MIN(valor), 2) as minimo,
           ROUND(MAX(valor), 2) as maximo
    FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
    GROUP BY planta, metrica
    ORDER BY planta, metrica
""").df()

## Gaps por fecha - días faltantes por mes

In [ ]:
# Verificar que cada mes tiene los días esperados
import calendar

dias_por_mes = con.execute("""
    SELECT mes, COUNT(DISTINCT fecha) as dias_con_datos
    FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
    GROUP BY mes
    ORDER BY mes
""").df()

dias_por_mes['dias_esperados'] = dias_por_mes['mes'].apply(lambda m: calendar.monthrange(2021, m)[1])
dias_por_mes['gap'] = dias_por_mes['dias_esperados'] - dias_por_mes['dias_con_datos']
dias_por_mes

## Detección de outliers (z-score) por planta y métrica

In [ ]:
# Outliers: z-score > 3 en valor, agrupado por planta + metrica
outliers = con.execute("""
    WITH stats AS (
        SELECT planta, metrica, 
               AVG(valor) as mu, 
               STDDEV(valor) as sigma
        FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
        WHERE valor IS NOT NULL
        GROUP BY planta, metrica
        HAVING STDDEV(valor) > 0
    )
    SELECT d.fecha, d.planta, d.metrica, d.valor,
           ROUND(s.mu, 2) as media,
           ROUND(ABS(d.valor - s.mu) / s.sigma, 2) as z_score
    FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true) d
    JOIN stats s ON d.planta = s.planta AND d.metrica = s.metrica
    WHERE d.valor IS NOT NULL
      AND ABS(d.valor - s.mu) / s.sigma > 3
    ORDER BY z_score DESC
""").df()

print(f"Outliers detectados (z > 3): {len(outliers)}")
outliers

## Nulos por planta y métrica

In [ ]:
nulos = con.execute("""
    SELECT planta, metrica, 
           COUNT(*) as total,
           SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) as nulos,
           ROUND(100.0 * SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_nulos
    FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
    GROUP BY planta, metrica
    HAVING SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) > 0
    ORDER BY nulos DESC
""").df()

print(f"Combinaciones planta-metrica con nulos: {len(nulos)}")
nulos

## Valores sospechosos (211.111...)

In [ ]:
# Buscar valores default sospechosos
sospechosos = con.execute("""
    SELECT fecha, planta, metrica, valor
    FROM read_parquet('../data/bronze/eersa_generacion/**/*.parquet', hive_partitioning=true)
    WHERE ABS(valor - 211.111) < 1
       OR ABS(valor - 5711.111) < 1
    ORDER BY fecha
""").df()

print(f"Valores sospechosos (≈211.111 o ≈5711.111): {len(sospechosos)}")
sospechosos